In [1]:
import sys
import sqlite3
import hashlib
import uuid
from pathlib import Path

# Configurar path del proyecto
PROJECT_ROOT = Path(r'D:\Master\TrabajoFinalUCM\TFM')
sys.path.insert(0, str(PROJECT_ROOT))

DB1_PATH = str(PROJECT_ROOT / 'data' / 'tui_recomendador.db')
DB2_PATH = str(PROJECT_ROOT / 'data' / 'tui_recomendador-javier.db')

print(f'DB1: {DB1_PATH}')
print(f'DB2: {DB2_PATH}')

DB1: D:\Master\TrabajoFinalUCM\TFM\data\tui_recomendador.db
DB2: D:\Master\TrabajoFinalUCM\TFM\data\tui_recomendador-javier.db


# Análisis de tui_recomendador.db

In [2]:
conn1 = sqlite3.connect(DB1_PATH)

total1 = conn1.execute('SELECT COUNT(*) FROM resenas').fetchone()[0]
print(f'Total registros: {total1}')

print('\nPor fuente:')
for f, n in conn1.execute('SELECT fuente, COUNT(*) FROM resenas GROUP BY fuente ORDER BY COUNT(*) DESC').fetchall():
    print(f'  {f}: {n}')

print('\nTop 15 destinos:')
for d, n in conn1.execute('SELECT destino_nombre, COUNT(*) FROM resenas GROUP BY destino_nombre ORDER BY COUNT(*) DESC LIMIT 15').fetchall():
    print(f'  {d}: {n}')

print('\nPor idioma:')
for i, n in conn1.execute('SELECT idioma, COUNT(*) FROM resenas GROUP BY idioma ORDER BY COUNT(*) DESC LIMIT 10').fetchall():
    print(f'  {i}: {n}')

conn1.close()

Total registros: 8952

Por fuente:
  reddit: 5177
  youtube: 3185
  tripadvisor: 584
  google_maps: 6

Top 15 destinos:
  Punta Cana: 239
  Bali: 199
  Antalya: 191
  Cuba: 190
  Cancun: 189
  Mallorca: 184
  Santorini: 182
  Tenerife: 179
  Hurghada: 179
  Riviera Maya: 177
  Jamaica: 162
  Ibiza: 161
  Vietnam Da Nang: 150
  Sri Lanka: 150
  Sardinia: 142

Por idioma:
  en: 4541
  es: 3734
  it: 141
  pt: 138
  de: 88
  ca: 86
  fr: 41
  tl: 17
  ro: 17
  id: 17


# Análisis de tui_recomendador de javier

In [3]:
conn2 = sqlite3.connect(DB2_PATH)

total2 = conn2.execute('SELECT COUNT(*) FROM resenas').fetchone()[0]
print(f'Total registros: {total2}')

print('\nPor fuente:')
for f, n in conn2.execute('SELECT fuente, COUNT(*) FROM resenas GROUP BY fuente ORDER BY COUNT(*) DESC').fetchall():
    print(f'  {f}: {n}')

print('\nTop 15 destinos:')
for d, n in conn2.execute('SELECT destino_nombre, COUNT(*) FROM resenas GROUP BY destino_nombre ORDER BY COUNT(*) DESC LIMIT 15').fetchall():
    print(f'  {d}: {n}')

print('\nPor idioma:')
for i, n in conn2.execute('SELECT idioma, COUNT(*) FROM resenas GROUP BY idioma ORDER BY COUNT(*) DESC LIMIT 10').fetchall():
    print(f'  {i}: {n}')

conn2.close()

Total registros: 6513

Por fuente:
  reddit: 4209
  youtube: 1790
  tripadvisor: 512
  google_maps: 2

Top 15 destinos:
  Tenerife: 180
  Ibiza: 178
  Santorini: 176
  Bulgaria: 171
  Bali: 170
  Riviera Maya: 166
  Jamaica: 166
  Las Vegas: 165
  Antalya: 164
  Cuba: 160
  Sri Lanka: 158
  Costa Rica: 153
  Sardinia: 152
  Gran Canaria: 150
  Mallorca: 149

Por idioma:
  en: 3559
  es: 2478
  it: 105
  pt: 90
  ca: 74
  de: 33
  fr: 28
  tl: 24
  ro: 13
  so: 12


# . Cruce y comparación 

In [4]:
# Calcular hashes de ambas BDs
def get_hashes(db_path):
    conn = sqlite3.connect(db_path)
    textos = conn.execute('SELECT texto_original FROM resenas WHERE texto_original IS NOT NULL').fetchall()
    hashes = set()
    for (t,) in textos:
        if t and len(t.strip()) > 10:
            hashes.add(hashlib.md5(t.strip().lower().encode()).hexdigest())
    conn.close()
    return hashes

hashes1 = get_hashes(DB1_PATH)
hashes2 = get_hashes(DB2_PATH)

comunes = hashes1 & hashes2
solo_db1 = hashes1 - hashes2
solo_db2 = hashes2 - hashes1
union = hashes1 | hashes2

print(f'Únicos en tui_recomendador.db:        {len(hashes1)}')
print(f'Únicos en tui_recomendador-javier.db: {len(hashes2)}')
print(f'En COMÚN (duplicados):                {len(comunes)}')
print(f'Solo en tui_recomendador.db:          {len(solo_db1)}')
print(f'Solo en javier.db:                    {len(solo_db2)}')
print(f'UNIÓN (total sin duplicados):         {len(union)}')
print(f'% de cruce:                           {len(comunes)/max(1,min(len(hashes1),len(hashes2)))*100:.1f}%')

print(f'\n--- RESUMEN ---')
print(f'Si unificamos: {len(union)} reseñas únicas')
print(f'Se ganarían {len(solo_db2)} reseñas nuevas de javier.db')
print(f'Se eliminarían {len(comunes)} duplicados')

Únicos en tui_recomendador.db:        8952
Únicos en tui_recomendador-javier.db: 6510
En COMÚN (duplicados):                3328
Solo en tui_recomendador.db:          5624
Solo en javier.db:                    3182
UNIÓN (total sin duplicados):         12134
% de cruce:                           51.1%

--- RESUMEN ---
Si unificamos: 12134 reseñas únicas
Se ganarían 3182 reseñas nuevas de javier.db
Se eliminarían 3328 duplicados


# Unificar left jin 

In [ ]:
# Ejecuta esta celda SOLO si quieres unificar

conn1 = sqlite3.connect(DB1_PATH)
conn2 = sqlite3.connect(DB2_PATH)

# Cargar hashes existentes en DB1
textos1 = conn1.execute('SELECT texto_original FROM resenas WHERE texto_original IS NOT NULL').fetchall()
hashes_db1 = set()
for (t,) in textos1:
    if t and len(t.strip()) > 10:
        hashes_db1.add(hashlib.md5(t.strip().lower().encode()).hexdigest())

# Leer reseñas de DB2
resenas2 = conn2.execute("""
    SELECT destino_nombre, fuente, texto_original, idioma, puntuacion,
           fecha_publicacion, url_fuente, fecha_extraccion
    FROM resenas WHERE texto_original IS NOT NULL
""").fetchall()

# Insertar solo las nuevas
insertadas = 0
for row in resenas2:
    destino, fuente, texto, idioma, puntuacion, fecha_pub, url, fecha_ext = row
    if not texto or len(texto.strip()) <= 10:
        continue
    h = hashlib.md5(texto.strip().lower().encode()).hexdigest()
    if h in hashes_db1:
        continue
    hashes_db1.add(h)
    conn1.execute(
        """INSERT INTO resenas (id_resena, destino_nombre, fuente, texto_original,
           idioma, puntuacion, fecha_publicacion, url_fuente, fecha_extraccion)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)""",
        (str(uuid.uuid4()), destino, fuente, texto, idioma, puntuacion, fecha_pub, url, fecha_ext)
    )
    insertadas += 1

conn1.commit()
total_final = conn1.execute('SELECT COUNT(*) FROM resenas').fetchone()[0]
conn1.close()
conn2.close()

print(f'Unificación completada')
print(f'   Reseñas nuevas insertadas: {insertadas}')
print(f'   Total final en tui_recomendador.db: {total_final}')

✅ Unificación completada
   Reseñas nuevas insertadas: 3182
   Total final en tui_recomendador.db: 12134
